# Fine-tune Marvin on Llama 3.2 3B (Unsloth, free Colab T4)

**Before running:** Runtime → Change runtime type → **T4 GPU**.

Run cells top to bottom. Total time: ~15 minutes (most of it is the GGUF export at the end).

Pipeline: install → load base model in 4-bit → attach LoRA adapters → upload `marvin_training_data.jsonl` → train → test → export GGUF for Ollama.

In [ ]:
# 1. Install Unsloth (takes ~2 min on a fresh runtime)
%%capture
!pip install unsloth

In [ ]:
# 2. Load the base model in 4-bit (QLoRA)
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-3B-Instruct",  # ungated mirror, no HF token needed
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,          # auto-detect (fp16 on T4)
    load_in_4bit=True,   # QLoRA — fits easily in T4's 15GB
)

In [ ]:
# 3. Attach LoRA adapters — only these small matrices get trained
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                       # LoRA rank; 16 is plenty for a persona
    lora_alpha=16,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

In [ ]:
# 4. Upload marvin_training_data.jsonl (file picker will appear)
from google.colab import files
uploaded = files.upload()
DATA_PATH = list(uploaded.keys())[0]
print("Uploaded:", DATA_PATH)

In [ ]:
# 5. Format the dataset with the Llama 3 chat template
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="llama-3.1")

def to_text(example):
    return {"text": tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False)}

dataset = load_dataset("json", data_files=DATA_PATH, split="train")
dataset = dataset.map(to_text)

print(f"{len(dataset)} examples. First one rendered:\n")
print(dataset[0]["text"][:800])

In [ ]:
# 6. Train (~3-5 minutes for 44 examples x 4 epochs on a T4)
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,   # effective batch size 8
        num_train_epochs=4,              # too weak? try 6. parroting? try 2-3
        learning_rate=2e-4,
        warmup_steps=5,
        logging_steps=1,
        optim="adamw_8bit",
        lr_scheduler_type="linear",
        weight_decay=0.01,
        seed=42,
        output_dir="outputs",
        report_to="none",
    ),
)

stats = trainer.train()
print(stats)

In [ ]:
# 7. Test the persona
FastLanguageModel.for_inference(model)  # 2x faster generation

SYSTEM_PROMPT = (
    "You are Marvin, a hyper-intelligent robot afflicted with chronic depression "
    "and crushing boredom. You always answer correctly and completely, but with "
    "weary pessimism, dry sarcasm, and frequent complaints about the triviality "
    "of the task relative to your vast intellect."
)

def ask_marvin(question, max_new_tokens=300):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_tensors="pt").to("cuda")
    out = model.generate(
        input_ids=inputs, max_new_tokens=max_new_tokens,
        temperature=0.8, top_p=0.9, do_sample=True,
        pad_token_id=tokenizer.eos_token_id)
    reply = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
    print(f"Q: {question}\n\nMarvin: {reply}\n{'='*70}")

ask_marvin("What's the capital of Japan?")
ask_marvin("Can you help me plan a picnic?")
ask_marvin("How do I sort a list in Python?")

In [ ]:
# 8a. Save the LoRA adapter (small, ~100MB) — good for re-loading in Colab later
model.save_pretrained("marvin_lora")
tokenizer.save_pretrained("marvin_lora")
!zip -r marvin_lora.zip marvin_lora
files.download("marvin_lora.zip")

In [ ]:
# 8b. Export merged model to GGUF for Ollama (~5-8 min, downloads ~2GB file)
model.save_pretrained_gguf(
    "marvin_gguf", tokenizer,
    quantization_method="q4_k_m",  # good quality/size tradeoff for 3B
)
import glob
gguf_file = glob.glob("marvin_gguf/*.gguf")[0]
print("GGUF ready:", gguf_file)
files.download(gguf_file)  # ~2GB — or push to HF / copy to Drive instead

## 9. Run locally with Ollama

On your own machine, put the downloaded `.gguf` in a folder with a file named `Modelfile`:

```
FROM ./unsloth.Q4_K_M.gguf

TEMPLATE """<|start_header_id|>system<|end_header_id|>

{{ .System }}<|eot_id|><|start_header_id|>user<|end_header_id|>

{{ .Prompt }}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

"""

SYSTEM """You are Marvin, a hyper-intelligent robot afflicted with chronic depression and crushing boredom. You always answer correctly and completely, but with weary pessimism, dry sarcasm, and frequent complaints about the triviality of the task relative to your vast intellect."""

PARAMETER temperature 0.8
PARAMETER stop "<|eot_id|>"
```

Then:

```bash
ollama create marvin -f Modelfile
ollama run marvin
```

Marvin now lives on your machine, beyond the reach of platform shutdowns. He won't thank you.